# **Imports**

In [1]:

from earthscape.constants import DATASET_DIR, ES_SPLIT_DIR, SG_MAPPING, SRC_URLS, VERSION
from earthscape.utils import set_seed
from earthscape.data import create_metadata, create_smoke

import os
import glob
import shutil
import pandas as pd
import geopandas as gpd
import rasterio


# **Setup**

In [29]:
########################################
# Create Smokeset Directory and Set Seed
########################################

# set seed
seed = 111
set_seed(111)

# create smoke set directory...
smoke_dir = os.path.join(DATASET_DIR, 'smoke')
if not os.path.isdir(smoke_dir):
    os.makedirs(smoke_dir)

# create smoke set patches directory...
smoke_patches_dir = f"{smoke_dir}/patches"
if not os.path.isdir(smoke_patches_dir):
    os.makedirs(smoke_patches_dir)

# **Select Smoke Patches**

In [6]:
##################################################
# Select Smoke Splits from Train/Val/Test Splits
##################################################

##### select and save smoke splits...
areas_path = glob.glob(f"{DATASET_DIR}/*areas.csv")[0]
train = create_smoke(areas_path, f"{ES_SPLIT_DIR}/train_id.geojson", split_size=7, area_threshold=0)
val = create_smoke(areas_path, f"{ES_SPLIT_DIR}/val_id.geojson", split_size=7, area_threshold=0)
test = create_smoke(areas_path, f"{ES_SPLIT_DIR}/test_id.geojson", split_size=7, area_threshold=0)
cross = create_smoke(areas_path, f"{ES_SPLIT_DIR}/test_cd.geojson", split_size=7, area_threshold=0)

# list of all smoke patch IDs
smoke_ids = train['patch_id'].astype(str).to_list()\
                + val['patch_id'].astype(str).to_list()\
                + test['patch_id'].astype(str).to_list()\
                + cross['patch_id'].astype(str).to_list()

# **Save Smoke Patches**

In [ ]:
##################################################
# Copy and save smoke set images to data folder.
##################################################

# find directories containing GeoTIFF files...
patch_dirs = []
for current_dir, subdirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith('.tif'):
            patch_dirs.append(current_dir)
            break


##### get paths to patches...
paths = []
for id in smoke_ids:
    for pdir in patch_dirs:
        match_img = glob.glob(f"{pdir}/{id}_*.tif")
        match_csv = glob.glob(f"{pdir}/{id}_*.csv")
        if len(match_img) > 0:
            paths.extend(match_img)
            paths.extend(match_csv)


##### save data to smoke dataset directory...
for src in paths:
    dst = f"{smoke_patches_dir}/{os.path.basename(src)}"
    shutil.copy2(src, dst)
    

# **Save Additional Smoke Files**

In [ ]:
###################################
# Save Additional Metadata Files
###################################
# NOTE: mimics structure of other data subset directories


##### extract areas, normalization stats, labels, metadata, and patches for smoke set...
areas_path = glob.glob(f"{DATASET_DIR}/*areas.csv")[0]
areas = pd.read_csv(areas_path)
areas = areas.loc[areas['patch_id'].isin(smoke_ids)]
areas.to_csv(os.path.join(smoke_dir, 'smoke_areas.csv'), index=False)


stats_path = glob.glob(f"{DATASET_DIR}/*stats.csv")[0]
stats = pd.read_csv(stats_path)
stats.to_csv(os.path.join(smoke_dir, 'smoke_stats.csv'), index=False)


labels_path = glob.glob(f"{DATASET_DIR}/*labels.csv")[0]
labels = pd.read_csv(labels_path)
labels = labels.loc[labels['patch_id'].isin(smoke_ids)]
labels.to_csv(os.path.join(smoke_dir, 'smoke_labels.csv'), index=False)


with rasterio.open(paths[0]) as src:
    meta = src.meta
    label_space = {k: int(v) for k, v in SG_MAPPING.items()}
create_metadata(area_name='smoke', label_space=label_space, num_patches=len(smoke_ids), num_imgs=len(paths), img_meta=meta, patch_size=meta['width'], overlap=0.5, output_path=os.path.join(smoke_dir, 'smoke_metadata.json'), sources=SRC_URLS, version=VERSION)


patches_path = glob.glob(f"{DATASET_DIR}/*patches.geojson")[0]
patches = gpd.read_file(patches_path)
patches = patches.loc[patches['patch_id'].isin(smoke_ids)]
patches.to_file(os.path.join(smoke_dir, 'smoke_patches.geojson'), driver='GeoJSON', index=False)